# SegFormer fine-tuning on Google Colab (T4)

This  will:
- install all required libraries
- download 500 images and masks from GitHub
- create an **empty mask** for every image without a labeled mask. Only a subset of images have masks. images without a mask are treated as **negative examples** with an all-zero mask
- fine-tune **SegFormer MIT-B1** by default (switch later to **MIT-B2**)
- evaluate the model and export sample predictions


In [ ]:
#@title 1. Install dependencies
!pip -q install -U transformers datasets evaluate accelerate huggingface_hub "pillow<12.0" matplotlib scikit-learn

In [ ]:
#@title 2. Imports and environment checks
import json
import os
import random
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Colab, go to Runtime -> Change runtime type -> T4 GPU.')


In [ ]:
#@title 3. Configuration
CHECKPOINT = 'nvidia/mit-b1'  # change to 'nvidia/mit-b2' later if wanted
OUTPUT_DIR = '/content/segformer-survey-maps-results'
DATA_ROOT = Path('/content/survey_maps_batch1')
IMAGES_DIR = DATA_ROOT / 'images'
MASKS_DIR = DATA_ROOT / 'masks'
GENERATED_MASKS_DIR = DATA_ROOT / 'generated_masks'

IMAGE_BASE_URL = 'https://raw.githubusercontent.com/rijpma/survey-maps/main/labelled/batch1/images'
MASK_BASE_URL = 'https://raw.githubusercontent.com/rijpma/survey-maps/main/labelled/batch1/masks'
GITHUB_API_IMAGES = 'https://api.github.com/repos/rijpma/survey-maps/contents/labelled/batch1/images'
GITHUB_API_MASKS = 'https://api.github.com/repos/rijpma/survey-maps/contents/labelled/batch1/masks'

IMAGE_SIZE = 512 #  switch to 384 for b2?
TEST_SIZE = 0.2
SEED = 42

NUM_EPOCHS = 20
TRAIN_BATCH_SIZE = 8 # switch to 4 for b2?
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 1

id2label = {0: 'background', 1: 'object'}
label2id = {v: k for k, v in id2label.items()}

DATA_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_MASKS_DIR.mkdir(parents=True, exist_ok=True)

print('Checkpoint:', CHECKPOINT)
print('Output dir:', OUTPUT_DIR)
print('Image size:', IMAGE_SIZE)


In [ ]:
#@title 4. Download images and masks from GitHub
import requests
from tqdm.auto import tqdm

def list_github_files(api_url):
    response = requests.get(api_url, timeout=60)
    response.raise_for_status()
    items = response.json()
    return [item['name'] for item in items if item['type'] == 'file' and item['name'].lower().endswith('.png')]

def download_files(file_names, base_url, destination_dir):
    destination_dir.mkdir(parents=True, exist_ok=True)
    for name in tqdm(file_names, desc=f'Downloading to {destination_dir.name}'):
        out_path = destination_dir / name
        if out_path.exists():
            continue
        url = f'{base_url}/{name}'
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        out_path.write_bytes(response.content)

image_names = sorted(list_github_files(GITHUB_API_IMAGES))
mask_names = sorted(list_github_files(GITHUB_API_MASKS))

print(f'Images on GitHub: {len(image_names)}')
print(f'Masks on GitHub: {len(mask_names)}')

download_files(image_names, IMAGE_BASE_URL, IMAGES_DIR)
download_files(mask_names, MASK_BASE_URL, MASKS_DIR)

print('Downloaded images:', len(list(IMAGES_DIR.glob('*.png'))))
print('Downloaded masks:', len(list(MASKS_DIR.glob('*.png'))))


In [ ]:
#@title 5. Build dataset records and generate empty masks where missing
def normalize_mask(mask_img):
    mask_arr = np.array(mask_img)
    if mask_arr.ndim == 3:
        mask_arr = mask_arr[..., 0]
    mask_arr = (mask_arr > 0).astype(np.uint8)
    return Image.fromarray(mask_arr, mode='L')

def make_empty_mask_like(image_path):
    img = Image.open(image_path)
    width, height = img.size
    return Image.fromarray(np.zeros((height, width), dtype=np.uint8), mode='L')

records = []
positive_count = 0
negative_count = 0

for image_path in sorted(IMAGES_DIR.glob('*.png')):
    mask_path = MASKS_DIR / image_path.name

    if mask_path.exists():
        normalized_mask = normalize_mask(Image.open(mask_path))
        generated_mask_path = GENERATED_MASKS_DIR / image_path.name
        normalized_mask.save(generated_mask_path)
        final_mask_path = generated_mask_path
        positive_count += 1
    else:
        empty_mask = make_empty_mask_like(image_path)
        generated_mask_path = GENERATED_MASKS_DIR / image_path.name
        empty_mask.save(generated_mask_path)
        final_mask_path = generated_mask_path
        negative_count += 1

    records.append({
        'image_path': str(image_path),
        'mask_path': str(final_mask_path),
        'has_object_mask': 1 if mask_path.exists() else 0,
        'file_name': image_path.name,
    })

print('Total records:', len(records))
print('Positive images with masks:', positive_count)
print('Negative images without masks:', negative_count)
assert len(records) > 0


In [ ]:
#@title 6. Stratified train/test split
indices = list(range(len(records)))
stratify_labels = [r['has_object_mask'] for r in records]

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=stratify_labels,
)

train_records = [records[i] for i in train_idx]
test_records = [records[i] for i in test_idx]

print('Train size:', len(train_records))
print('Test size:', len(test_records))
print('Train positives:', sum(r['has_object_mask'] for r in train_records))
print('Test positives:', sum(r['has_object_mask'] for r in test_records))


In [ ]:
#@title 7. Create Hugging Face datasets
train_ds = Dataset.from_list(train_records)
test_ds = Dataset.from_list(test_records)
raw_datasets = DatasetDict({'train': train_ds, 'test': test_ds})
raw_datasets


In [ ]:
#@title 8. Visual sanity check
def show_samples(dataset, n=4):
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(10, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row in range(n):
        example = dataset[row]
        image = Image.open(example['image_path']).convert('RGB')
        mask = Image.open(example['mask_path'])
        mask_np = np.array(mask)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(example['file_name'])
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title(f"Mask (has_object={example['has_object_mask']})")
        axes[row, 1].axis('off')

        overlay = np.array(image).copy()
        overlay_mask = mask_np > 0
        overlay[overlay_mask] = [255, 0, 0]
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title('Overlay')
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

show_samples(raw_datasets['train'], n=4)


In [ ]:
#@title 9. Processor and transforms
processor = SegformerImageProcessor.from_pretrained(
    CHECKPOINT,
    do_resize=True,
    size={'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    do_reduce_labels=False,
)

def load_image_and_mask(example):
    image = Image.open(example['image_path']).convert('RGB')
    mask = Image.open(example['mask_path'])
    mask_arr = np.array(mask)
    if mask_arr.ndim == 3:
        mask_arr = mask_arr[..., 0]
    mask_arr = (mask_arr > 0).astype(np.uint8)
    return image, mask_arr

def train_transforms(example_batch):
    images = []
    labels = []
    for image_path, mask_path in zip(example_batch['image_path'], example_batch['mask_path']):
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path)
        mask_arr = np.array(mask)
        if mask_arr.ndim == 3:
            mask_arr = mask_arr[..., 0]
        mask_arr = (mask_arr > 0).astype(np.uint8)
        images.append(image)
        labels.append(mask_arr)

    inputs = processor(images=images, segmentation_maps=labels, return_tensors='pt')
    return inputs

transformed_datasets = raw_datasets.with_transform(train_transforms)


In [ ]:
#@title 10. Load model
model = SegformerForSemanticSegmentation.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable_params:,}')
print(f'Total params: {total_params:,}')


In [ ]:
#@title 11. Metrics and trainer helpers
metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.from_numpy(logits)

    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=labels.shape[-2:],
        mode='bilinear',
        align_corners=False,
    )
    pred_labels = upsampled_logits.argmax(dim=1).cpu().numpy()

    metrics = metric.compute(
        predictions=pred_labels,
        references=labels,
        num_labels=2,
        ignore_index=255,
        reduce_labels=False,
    )

    return {
        'mean_iou': metrics['mean_iou'],
        'mean_accuracy': metrics['mean_accuracy'],
        'iou_background': metrics['per_category_iou'][0],
        'iou_object': metrics['per_category_iou'][1],
    }

class ValidationLogger(TrainerCallback):
    def __init__(self, log_path='eval_results.jsonl'):
        self.log_path = log_path

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'eval_loss' in logs:
            with open(self.log_path, 'a') as f:
                payload = {'epoch': state.epoch, **logs}
                f.write(json.dumps(payload) + '\n')


In [ ]:
#@title 12. Training arguments for Colab T4
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model='mean_iou',
    greater_is_better=True,
    dataloader_num_workers=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
    bf16=False,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=transformed_datasets['train'],
    eval_dataset=transformed_datasets['test'],
    compute_metrics=compute_metrics,
    callbacks=[ValidationLogger()],
)

print(training_args)


In [ ]:
#@title 13. Start training
print('Starting training on device:', trainer.args.device)
train_result = trainer.train()
train_result


In [ ]:
#@title 14. Final evaluation
eval_metrics = trainer.evaluate()
eval_metrics


In [ ]:
#@title 15. Save best model
final_model_path = os.path.join(OUTPUT_DIR, 'final_best_model')
trainer.save_model(final_model_path)
processor.save_pretrained(final_model_path)
print('Best model saved to:', final_model_path)


In [ ]:
#@title 16. Export sample predictions
PREDICTION_DIR = Path(OUTPUT_DIR) / 'prediction_samples'
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def predict_mask(image):
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    upsampled_logits = torch.nn.functional.interpolate(
        outputs.logits,
        size=image.size[::-1],
        mode='bilinear',
        align_corners=False,
    )
    prediction = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    return prediction

num_examples = min(8, len(test_records))
chosen_examples = random.sample(test_records, num_examples)

for i, example in enumerate(chosen_examples):
    image = Image.open(example['image_path']).convert('RGB')
    gt_mask = np.array(Image.open(example['mask_path']))
    pred_mask = predict_mask(image)

    image.save(PREDICTION_DIR / f'{i:02d}_image.png')
    Image.fromarray((gt_mask > 0).astype(np.uint8) * 255).save(PREDICTION_DIR / f'{i:02d}_gt.png')
    Image.fromarray(pred_mask * 255).save(PREDICTION_DIR / f'{i:02d}_pred.png')

    image_np = np.array(image).astype(np.float32)
    overlay = image_np.copy()
    red = np.array([255, 0, 0], dtype=np.float32)
    alpha = 0.7
    mask = pred_mask > 0
    overlay[mask] = (1 - alpha) * image_np[mask] + alpha * red
    overlay = overlay.astype(np.uint8)
    Image.fromarray(overlay).save(PREDICTION_DIR / f'{i:02d}_overlay.png')

print('Saved prediction samples to:', PREDICTION_DIR)


In [ ]:
#@title 17. Preview prediction samples
sample_overlays = sorted(PREDICTION_DIR.glob('*_overlay.png'))[:4]
fig, axes = plt.subplots(len(sample_overlays), 2, figsize=(12, 4 * max(1, len(sample_overlays))))
if len(sample_overlays) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, overlay_path in enumerate(sample_overlays):
    original_path = PREDICTION_DIR / overlay_path.name.replace('_overlay.png', '_image.png')

    axes[row, 0].imshow(Image.open(original_path))
    axes[row, 0].set_title(original_path.name)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(Image.open(overlay_path))
    axes[row, 1].set_title(overlay_path.name)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()
